# COSC2753 | Machine Learning

## Task 3: Occasion and Gender Classification

# 1. Introduction

Task 3 trains two independent classifiers: `gender` and occasion (`usage`). Keeping them separate matches the required prediction columns, avoids a sparse Cartesian label space, and permits target-specific missing-label and imbalance handling. **Notebook outputs are intentionally cleared. The existing checkpoints are prototypes used for application testing, not the final investigation. Member 4 must run both controlled candidates for each target, complete every analysis prompt, and replace both mock checkpoints.** Both targets must be developed, evaluated, and compared separately in this notebook.

# 2. Library Imports and Setup

In [ ]:
from copy import deepcopy
from pathlib import Path
import json, sys, time

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from app.server.utils.classifier import FashionClassifier
from app.server.utils.handcrafted import DEFAULT_FEATURE_CONFIG, handcrafted_feature
from evaluation import calibration_table, expected_calibration_error, high_confidence_errors, ordered_estimator_probabilities, saved_model_robustness, select_validation_candidate, subgroup_metrics
from preprocessing import IMAGE_SIZE, NORMALISATION_PATH, SEED, seed_everything, select_torch_device, task_frame
seed_everything(SEED); torch.manual_seed(SEED)
DEVICE = select_torch_device()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
with NORMALISATION_PATH.open(encoding='utf-8') as handle:
    normalisation = json.load(handle)
def supported_macro_f1(truth, predictions):
    return f1_score(truth, predictions, labels=np.unique(truth), average='macro', zero_division=0)
DEVICE

# 3. Shared Split and Target-Specific Filtering

In [ ]:
frames = {
    target: {split: task_frame(target, split) for split in ('train', 'validation', 'test')}
    for target in ('gender', 'usage')
}
pd.DataFrame({
    (target, split): frame[target].value_counts()
    for target, split_frames in frames.items() for split, frame in split_frames.items()
}).fillna(0).astype(int)

A genuinely blank target is excluded only from that target's rows. The literal string `NA` remains a valid usage class until its meaning is confirmed. Both targets retain the same product-group split assignment.

# 4. Gender and Usage EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for target, axis in zip(('gender', 'usage'), axes):
    counts = frames[target]['train'][target].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axis)
    axis.set_title(f'{target} training distribution')
    axis.tick_params(axis='x', rotation=35)
plt.tight_layout()

In [ ]:
pd.crosstab(frames['usage']['train']['articleType'], frames['usage']['train']['usage'], normalize='index').head(25).style.background_gradient(cmap='Greens')

## 4.1 Observations

For gender, discuss imbalance, unisex presentation, label overlap, and article-type dependence without implying that image appearance defines a person's gender. For usage, discuss Casual dominance, rare labels, visual ambiguity, and the literal-`NA` policy.

# 5. Majority Baselines

In [ ]:
baseline_rows, majority_models, majority_probabilities = [], {}, {}
for target in ('gender', 'usage'):
    train_frame, validation_frame = frames[target]['train'], frames[target]['validation']
    dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(train_frame), 1)), train_frame[target])
    labels = sorted(train_frame[target].unique())
    prediction = dummy.predict(np.zeros((len(validation_frame), 1)))
    majority_models[target] = dummy
    majority_probabilities[target] = ordered_estimator_probabilities(dummy, np.zeros((len(validation_frame), 1)), labels)
    baseline_rows.append({
        'target': target, 'accuracy': accuracy_score(validation_frame[target], prediction),
        'macro_f1': supported_macro_f1(validation_frame[target], prediction),
    })
baseline_results = pd.DataFrame(baseline_rows).set_index('target')
baseline_results

# 6. HOG + HSV Classical Baselines

In [ ]:
feature_cache = {}
def handcrafted_feature_row(row):
    if row.id in feature_cache: return feature_cache[row.id]
    with Image.open(row.image_path) as source:
        feature_cache[row.id] = handcrafted_feature(source, DEFAULT_FEATURE_CONFIG)
    return feature_cache[row.id]

classical_rows, classical_models, classical_probabilities = [], {}, {}
for target in ('gender', 'usage'):
    train_frame, validation_frame = frames[target]['train'], frames[target]['validation']
    train_features = np.vstack([handcrafted_feature_row(row) for row in train_frame.itertuples()])
    validation_features = np.vstack([handcrafted_feature_row(row) for row in validation_frame.itertuples()])
    baseline = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs'),
    ).fit(train_features, train_frame[target])
    labels = sorted(train_frame[target].unique())
    prediction = baseline.predict(validation_features)
    classical_models[target] = baseline
    classical_probabilities[target] = ordered_estimator_probabilities(baseline, validation_features, labels)
    classical_rows.append({'target': target, 'accuracy': accuracy_score(validation_frame[target], prediction), 'macro_f1': supported_macro_f1(validation_frame[target], prediction)})
classical_results = pd.DataFrame(classical_rows).set_index('target')
classical_results

# 7. Shared Image Pipeline

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])), transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(8, translate=(0.05, 0.05)), transforms.ColorJitter(0.12, 0.12),
    transforms.ToTensor(), transforms.Normalize(normalisation['mean'], normalisation['std']),
])
evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])), transforms.ToTensor(),
    transforms.Normalize(normalisation['mean'], normalisation['std']),
])

class FashionDataset(Dataset):
    def __init__(self, frame, target, label_to_index, transform):
        self.frame, self.target = frame.reset_index(drop=True), target
        self.label_to_index, self.transform = label_to_index, transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, self.label_to_index[row[self.target]]

# 8. Compact Residual CNN From Scratch

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(output_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity() if input_channels == output_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 1, stride, bias=False), nn.BatchNorm2d(output_channels),
        )
    def forward(self, inputs):
        residual = self.shortcut(inputs)
        outputs = self.relu(self.bn1(self.conv1(inputs)))
        outputs = self.bn2(self.conv2(outputs))
        return self.relu(outputs + residual)

class CompactCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32), nn.MaxPool2d(2), ResidualBlock(32, 64), nn.MaxPool2d(2),
            ResidualBlock(64, 128), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, inputs): return self.classifier(self.features(inputs))

# 9. Reusable Training Function

The function trains one independently initialized model and selects its checkpoint only by validation macro F1.

In [ ]:
def train_target(target, loss_mode='weighted', epochs=40, patience_limit=7):
    if loss_mode not in {'ordinary', 'weighted', 'class_balanced'}:
        raise ValueError('loss_mode must be ordinary, weighted, or class_balanced')
    seed_everything(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    train_frame, validation_frame = frames[target]['train'], frames[target]['validation']
    labels = sorted(train_frame[target].unique())
    label_to_index = {label: index for index, label in enumerate(labels)}
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(FashionDataset(train_frame, target, label_to_index, train_transform), 64, shuffle=True, num_workers=0, generator=generator)
    validation_loader = DataLoader(FashionDataset(validation_frame, target, label_to_index, evaluation_transform), 128, num_workers=0)
    model = CompactCNN(len(labels)).to(DEVICE)
    counts = train_frame[target].value_counts().reindex(labels).values
    if loss_mode == 'ordinary':
        weights = None
    elif loss_mode == 'class_balanced':
        beta = 0.9999; values = (1.0 - beta) / (1.0 - np.power(beta, counts)); values /= values.mean()
        weights = torch.tensor(values, dtype=torch.float32, device=DEVICE)
    else:
        weights = torch.tensor(len(train_frame) / (len(labels) * counts), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    def epoch(loader, training=False):
        model.train(training); losses, truth, predictions = [], [], []
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                logits = model(images); loss = criterion(logits, targets)
                if training: loss.backward(); optimizer.step()
            losses.append(loss.item() * len(targets)); truth.extend(targets.cpu().tolist()); predictions.extend(logits.argmax(1).cpu().tolist())
        return {'loss': sum(losses) / len(loader.dataset), 'accuracy': accuracy_score(truth, predictions), 'macro_f1': supported_macro_f1(truth, predictions)}

    history, best_state, best_f1, patience = [], None, -1.0, 0
    for epoch_number in range(1, epochs + 1):
        started = time.perf_counter(); train_metrics = epoch(train_loader, True); validation_metrics = epoch(validation_loader)
        history.append({'epoch': epoch_number, 'loss_mode': loss_mode, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'validation_{k}': v for k, v in validation_metrics.items()}})
        print(target, loss_mode, epoch_number, validation_metrics, f'{time.perf_counter() - started:.1f}s')
        if validation_metrics['macro_f1'] > best_f1:
            best_f1, best_state, patience = validation_metrics['macro_f1'], deepcopy(model.state_dict()), 0
        else:
            patience += 1
            if patience >= patience_limit: break
    model.load_state_dict(best_state)
    return model.cpu(), labels, label_to_index, pd.DataFrame(history), best_f1

# 10. Gender Classifier

In [ ]:
@torch.inference_mode()
def compare_and_select_target(target, runs):
    first_result = next(iter(runs.values()))
    labels, label_to_index = first_result[1], first_result[2]
    validation_frame = frames[target]['validation']
    validation_truth = np.asarray([label_to_index[label] for label in validation_frame[target]])
    loader = DataLoader(FashionDataset(validation_frame, target, label_to_index, evaluation_transform), 128, num_workers=0)
    rows = [
        {'method': 'majority', 'model_type': 'reference_only', 'validation_accuracy': baseline_results.loc[target, 'accuracy'], 'validation_macro_f1': baseline_results.loc[target, 'macro_f1'], 'validation_ece': expected_calibration_error(validation_truth, majority_probabilities[target]), 'complexity_parameters': 1, 'epochs_run': 0, 'eligible_for_selection': False},
        {'method': 'hog_hsv_logistic_regression', 'model_type': 'hog_hsv_logistic_regression', 'validation_accuracy': classical_results.loc[target, 'accuracy'], 'validation_macro_f1': classical_results.loc[target, 'macro_f1'], 'validation_ece': expected_calibration_error(validation_truth, classical_probabilities[target]), 'complexity_parameters': classical_models[target][-1].coef_.size + classical_models[target][-1].intercept_.size, 'epochs_run': 0, 'eligible_for_selection': True},
    ]
    for loss_mode, result in runs.items():
        cnn = result[0].to(DEVICE).eval(); values = []
        for images, _ in loader:
            values.append(cnn(images.to(DEVICE)).softmax(1).cpu().numpy())
        probabilities = np.vstack(values); result[0].cpu()
        rows.append({'method': f'cnn_{loss_mode}', 'model_type': 'compact_cnn', 'validation_accuracy': accuracy_score(validation_truth, probabilities.argmax(1)), 'validation_macro_f1': result[4], 'validation_ece': expected_calibration_error(validation_truth, probabilities), 'complexity_parameters': sum(parameter.numel() for parameter in result[0].parameters()), 'epochs_run': len(result[3]), 'eligible_for_selection': True})
    comparison = pd.DataFrame(rows).set_index('method')
    method = select_validation_candidate(comparison.loc[comparison.eligible_for_selection])
    model_type = comparison.loc[method, 'model_type']
    if model_type == 'compact_cnn':
        loss_mode = method.removeprefix('cnn_')
        model, labels, label_to_index, history, _ = runs[loss_mode]
    else:
        loss_mode, model = None, classical_models[target]
        history = pd.DataFrame([{'method': method, **comparison.loc[method].to_dict()}])
    return {'method': method, 'model_type': model_type, 'loss_mode': loss_mode, 'model': model, 'labels': labels, 'mapping': label_to_index, 'history': history, 'best_f1': float(comparison.loc[method, 'validation_macro_f1']), 'comparison': comparison}

gender_runs = {mode: train_target('gender', mode) for mode in ('ordinary', 'weighted')}
gender_selection = compare_and_select_target('gender', gender_runs)
gender_model, gender_labels, gender_mapping = gender_selection['model'], gender_selection['labels'], gender_selection['mapping']
gender_history, gender_best_f1, gender_comparison = gender_selection['history'], gender_selection['best_f1'], gender_selection['comparison']
GENDER_SELECTED_METHOD, GENDER_SELECTED_MODEL_TYPE, GENDER_SELECTED_LOSS_MODE = gender_selection['method'], gender_selection['model_type'], gender_selection['loss_mode']
display(gender_comparison)
print(f'Selected gender method {GENDER_SELECTED_METHOD!r} before internal-test evaluation.')
if 'epoch' in gender_history.columns: gender_history.plot(x='epoch', y=['train_macro_f1', 'validation_macro_f1'], figsize=(8, 4), title='Selected gender learning curve'); plt.show()

## 9.1 Gender analysis

Compare ordinary and weighted cross-entropy under identical settings. Discuss convergence, overlapping presentation, subgroup performance by article type, calibration, robustness, and responsible interpretation.

# 11. Occasion/Usage Classifier

In [ ]:
usage_runs = {mode: train_target('usage', mode) for mode in ('ordinary', 'class_balanced')}
usage_selection = compare_and_select_target('usage', usage_runs)
usage_model, usage_labels, usage_mapping = usage_selection['model'], usage_selection['labels'], usage_selection['mapping']
usage_history, usage_best_f1, usage_comparison = usage_selection['history'], usage_selection['best_f1'], usage_selection['comparison']
USAGE_SELECTED_METHOD, USAGE_SELECTED_MODEL_TYPE, USAGE_SELECTED_LOSS_MODE = usage_selection['method'], usage_selection['model_type'], usage_selection['loss_mode']
display(usage_comparison)
print(f'Selected usage method {USAGE_SELECTED_METHOD!r} before internal-test evaluation.')
if 'epoch' in usage_history.columns: usage_history.plot(x='epoch', y=['train_macro_f1', 'validation_macro_f1'], figsize=(8, 4), title='Selected usage learning curve'); plt.show()

## 10.1 Usage analysis

Compare ordinary and weighted loss, then run a clearly labelled sensitivity analysis for literal `NA` without silently changing the final vocabulary. Discuss rare labels, Casual dominance, subjectivity, and article-type shortcuts.

# 12. One-Time Internal-Test Evaluation

In [ ]:
@torch.inference_mode()
def evaluate_target(target, model, model_type, labels, label_to_index):
    test_frame = frames[target]['test']
    truth = np.asarray([label_to_index[label] for label in test_frame[target]])
    if model_type == 'compact_cnn':
        loader = DataLoader(FashionDataset(test_frame, target, label_to_index, evaluation_transform), 128, num_workers=0)
        model = model.to(DEVICE).eval(); values = []
        for images, _ in loader:
            values.append(model(images.to(DEVICE)).softmax(1).cpu().numpy())
        probabilities, model = np.vstack(values), model.cpu()
    else:
        features = np.vstack([handcrafted_feature_row(row) for row in test_frame.itertuples()])
        probabilities = ordered_estimator_probabilities(model, features, labels)
    predictions = probabilities.argmax(1)
    metrics = {'accuracy': accuracy_score(truth, predictions), 'macro_f1': supported_macro_f1(truth, predictions)}
    return metrics, truth, predictions, probabilities, model

gender_metrics, gender_truth, gender_predictions, gender_probabilities, gender_model = evaluate_target('gender', gender_model, GENDER_SELECTED_MODEL_TYPE, gender_labels, gender_mapping)
usage_metrics, usage_truth, usage_predictions, usage_probabilities, usage_model = evaluate_target('usage', usage_model, USAGE_SELECTED_MODEL_TYPE, usage_labels, usage_mapping)
pd.DataFrame([gender_metrics, usage_metrics], index=['gender', 'usage'])

In [ ]:
for target, truth, predictions, probabilities, labels in [
    ('gender', gender_truth, gender_predictions, gender_probabilities, gender_labels),
    ('usage', usage_truth, usage_predictions, usage_probabilities, usage_labels),
]:
    print(f'\n{target.upper()}')
    label_indices = np.arange(len(labels))
    print(classification_report(
        truth, predictions, labels=label_indices, target_names=labels, zero_division=0,
    ))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay.from_predictions(
        truth, predictions, labels=label_indices, display_labels=labels, normalize='true',
        ax=ax, cmap='Blues', xticks_rotation=45,
    )
    plt.show()
    display(calibration_table(truth, probabilities))
    display(high_confidence_errors(frames[target]['test'], target, labels, truth, predictions, probabilities))
    truth_labels = [labels[index] for index in truth]
    prediction_labels = [labels[index] for index in predictions]
    display(subgroup_metrics(frames[target]['test'], truth_labels, prediction_labels, group_column='articleType', minimum_support=30))

## 11.1 Comparative failure analysis

Compare the two targets without averaging them into one score. Include high-confidence errors, article-type subgroups, per-class support, calibration, mild corruption robustness, latency, model size, and qualitative ambiguity.

# 13. Save Both Selected Models

In [ ]:
def save_classifier(target, model, model_type, labels, method, loss_mode, comparison, test_metrics):
    path = ROOT / 'models' / f'{target}_model.pt'
    validation_metrics = {key: float(comparison.loc[method, key]) for key in ('validation_accuracy', 'validation_macro_f1', 'validation_ece')}
    if model_type == 'compact_cnn':
        checkpoint = {'model_type': model_type, 'target': target, 'labels': labels, 'state_dict': model.state_dict(), 'mean': normalisation['mean'], 'std': normalisation['std'], 'image_size': list(IMAGE_SIZE), 'dropout': 0.2, 'loss_mode': loss_mode}
    else:
        checkpoint = {'model_type': model_type, 'target': target, 'labels': labels, 'estimator': model, 'feature_config': DEFAULT_FEATURE_CONFIG}
    checkpoint.update({'selection_metric': 'validation_macro_f1', 'best_validation_macro_f1': validation_metrics['validation_macro_f1'], 'validation_metrics': validation_metrics, 'test_metrics': test_metrics, 'seed': SEED})
    torch.save(checkpoint, path)
    return path

gender_path = save_classifier('gender', gender_model, GENDER_SELECTED_MODEL_TYPE, gender_labels, GENDER_SELECTED_METHOD, GENDER_SELECTED_LOSS_MODE, gender_comparison, gender_metrics)
usage_path = save_classifier('usage', usage_model, USAGE_SELECTED_MODEL_TYPE, usage_labels, USAGE_SELECTED_METHOD, USAGE_SELECTED_LOSS_MODE, usage_comparison, usage_metrics)
gender_robustness = saved_model_robustness(FashionClassifier(gender_path, device='cpu'), frames['gender']['test'], 'gender', SEED)
usage_robustness = saved_model_robustness(FashionClassifier(usage_path, device='cpu'), frames['usage']['test'], 'usage', SEED)
display(gender_robustness)
display(usage_robustness)
print(f'Gender checkpoint: {gender_path.stat().st_size / 1024**2:.2f} MiB | Usage checkpoint: {usage_path.stat().st_size / 1024**2:.2f} MiB')
gender_history.to_csv(ROOT / 'models' / 'gender_history.csv', index=False)
usage_history.to_csv(ROOT / 'models' / 'usage_history.csv', index=False)
gender_comparison.to_csv(ROOT / 'models' / 'gender_comparison.csv')
usage_comparison.to_csv(ROOT / 'models' / 'usage_comparison.csv')
gender_path, usage_path

# 14. Final Prediction and Submission

Verify a single target with `python scripts/task3_occasion_gender_classification.py --image path/to/image.jpg --target gender`. After Tasks 1 and 2 deliver their checkpoints, generate all four prediction columns with `python scripts/task3_occasion_gender_classification.py --submission`.

# 15. Ultimate Judgement and Conclusion

Replace this prompt with separate comparisons among majority, HOG+HSV, ordinary-loss residual CNN, and weighted residual CNN for both targets; include the final literal-`NA` policy, major limitations, and confirmation that both checkpoints satisfy the shared inference contract.